### Uso del motor de inferencia desde un Notebook o script Python

In [1]:
from moodle_log_adm.core.inference import InferenceEngine
from moodle_log_adm.core.collections import LogDict, RecordFile, TemplateFile
from moodle_log_adm.core.loading import RECORD_FILE_LOADERS, TEMPLATE_FILE_LOADERS

# Instancia del TemplateBuilder
template_builder = InferenceEngine.from_yaml("config.yaml").get_template_builder()

# Carga de un fichero de logs crudos en formato CSV
record_file_csv = RecordFile(
    "logs.csv",
    RECORD_FILE_LOADERS[".csv"],
    ("Component", "Event name", "Description"),
)

# Carga de un fichero de logs crudos en formato JSON
record_file_json = RecordFile(
    "logs.json",
    RECORD_FILE_LOADERS[".json"],
    ("component", "eventname", "description"),
)

# Carga de un fichero de templates en formato JSON
template_file_json = TemplateFile(
    "templates.json",
    TEMPLATE_FILE_LOADERS[".json"],
)

# Construcción de los diccionarios
master_success_dict = LogDict()
master_failure_dict = LogDict()

# Extracción de templates y fusión de diccionarios
for source in (record_file_csv, record_file_json, template_file_json):
    success_dict, failure_dict = source.extract_templates(template_builder)
    master_success_dict += success_dict
    master_failure_dict += failure_dict

# Resumen de resultados
print(f"""
Success logs count: {len(master_success_dict)}
Failure logs count: {len(master_failure_dict)}

Success logs sample:
{"\n".join(str(master_success_dict).splitlines()[:10])}

Failure logs sample:
{"\n".join(str(master_failure_dict).splitlines()[:10])}
""")


Success logs count: 236
Failure logs count: 0

Success logs sample:
Activity report:
  Activity report viewed:
  - The user with id '(?<userId>-?\d+)?' viewed the outline activity report for the course with id '(?<courseId>-?\d+)?'.
  Outline report viewed:
  - The user with id '(?<userId>-?\d+)?' viewed the outline report for the user with id '(?<userId>-?\d+)?' for the course with id '(?<courseId>-?\d+)?'.
Assignment:
  A submission has been submitted.:
  - The user with id '(?<userId>-?\d+)?' has submitted the submission with id '(?<submissionId>-?\d+)?' for the assignment with course module id '(?<moduleId>-?\d+)?'.
  All the submissions are being downloaded.:
  - The user with id '(?<userId>-?\d+)?' has downloaded all the submissions for the assignment with course module id '(?<moduleId>-?\d+)?'.

Failure logs sample:
{}

